<a href="https://colab.research.google.com/github/abishekksp14/outdoor-expedition-AI-agent/blob/main/Outdoor%20expedition%20AI%20agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install requests google-generativeai pandas gradio --quiet

In [2]:
GOOGLE_AI_API_KEY = None
try:
    from google.colab import userdata
    GOOGLE_AI_API_KEY = userdata.get("GOOGLE_AI_API_KEY")
    print("✅ Loaded Gemini API key from Colab Secrets.")
except Exception:
    pass

if not GOOGLE_AI_API_KEY:
    GOOGLE_AI_API_KEY = input("Couldn't find a Colab Secret named GOOGLE_AI_API_KEY — paste your free Gemini API key here: ").strip()

✅ Loaded Gemini API key from Colab Secrets.


In [3]:
import requests

WEATHER_CODES = {
    0: "clear sky", 1: "mainly clear", 2: "partly cloudy", 3: "overcast",
    45: "fog", 48: "depositing rime fog",
    51: "light drizzle", 53: "moderate drizzle", 55: "dense drizzle",
    61: "slight rain", 63: "moderate rain", 65: "heavy rain",
    71: "slight snow", 73: "moderate snow", 75: "heavy snow",
    80: "slight rain showers", 81: "moderate rain showers", 82: "violent rain showers",
    95: "thunderstorm", 96: "thunderstorm with hail", 99: "thunderstorm with heavy hail",
}

def _to_plain(obj):
    """Recursively convert Gemini's proto MapComposite/RepeatedComposite objects into plain
    dicts/lists so printed output and the Gradio UI show readable data instead of object reprs."""
    if hasattr(obj, "items"):
        return {k: _to_plain(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)) or (hasattr(obj, "__iter__") and not isinstance(obj, (str, bytes, dict))):
        try:
            return [_to_plain(v) for v in obj]
        except TypeError:
            return obj
    return obj

def _aqi_label(aqi):
    if aqi <= 50: return "Good"
    if aqi <= 100: return "Moderate"
    if aqi <= 150: return "Unhealthy for Sensitive Groups"
    if aqi <= 200: return "Unhealthy"
    if aqi <= 300: return "Very Unhealthy"
    return "Hazardous"

def _geocode_query(query: str) -> list:
    resp = requests.get("https://geocoding-api.open-meteo.com/v1/search", params={"name": query, "count": 5})
    resp.raise_for_status()
    data = resp.json()
    matches = []
    for r in (data.get("results") or []):
        matches.append({
            "name": r["name"],
            "country": r.get("country"),
            "admin1": r.get("admin1"),
            "latitude": r["latitude"],
            "longitude": r["longitude"],
        })
    return matches

def resolve_location(place_name: str) -> dict:
    """Look up latitude/longitude for a place name using free geocoding. Call this first for
    EACH candidate location if you only have city names, not coordinates. Works best with just a
    city name (e.g. 'Denver'); if given 'City, State' or 'City, Country' it will automatically
    retry with just the city part."""
    matches = _geocode_query(place_name)
    if not matches and "," in place_name:
        simplified = place_name.split(",")[0].strip()
        matches = _geocode_query(simplified)
        if matches:
            return {"matches": matches, "note": f"No results for '{place_name}' as given; retried successfully with '{simplified}'."}
    if not matches:
        return {"matches": [], "note": f"No results found for '{place_name}'. Try a simpler query, e.g. just the city name."}
    return {"matches": matches}

def get_daily_forecast(latitude: float, longitude: float, forecast_days: int = 7) -> dict:
    """Get the real daily weather forecast for the next N days (max 16): max/min temp (F), rain
    chance, max wind, UV index, and sunrise/sunset times. Call this first for a location to see
    the whole available window before narrowing in on a specific day."""
    forecast_days = max(1, min(16, int(round(float(forecast_days)))))
    resp = requests.get("https://api.open-meteo.com/v1/forecast", params={
        "latitude": latitude, "longitude": longitude,
        "daily": "temperature_2m_max,temperature_2m_min,precipitation_probability_max,windspeed_10m_max,uv_index_max,weathercode,sunrise,sunset",
        "temperature_unit": "fahrenheit",
        "windspeed_unit": "mph",
        "timezone": "auto",
        "forecast_days": forecast_days,
    })
    resp.raise_for_status()
    daily = resp.json()["daily"]
    days = []
    for i, date in enumerate(daily["time"]):
        days.append({
            "date": date,
            "max_temp_f": daily["temperature_2m_max"][i],
            "min_temp_f": daily["temperature_2m_min"][i],
            "precip_probability_pct": daily["precipitation_probability_max"][i],
            "max_wind_mph": daily["windspeed_10m_max"][i],
            "uv_index_max": daily["uv_index_max"][i],
            "conditions": WEATHER_CODES.get(daily["weathercode"][i], "unknown"),
            "sunrise": daily["sunrise"][i],
            "sunset": daily["sunset"][i],
        })
    return {"days": days}

def get_hourly_forecast_for_day(latitude: float, longitude: float, date: str) -> dict:
    """Get an hour-by-hour forecast for one specific date (YYYY-MM-DD). Use this to self-verify a
    promising day BEFORE finalizing a recommendation — daily averages can hide bad windows within
    the day, e.g. a high rain chance that's actually all before 8am, or a temperature spike that
    only happens in the afternoon."""
    resp = requests.get("https://api.open-meteo.com/v1/forecast", params={
        "latitude": latitude, "longitude": longitude,
        "hourly": "temperature_2m,precipitation_probability,windspeed_10m,weathercode",
        "temperature_unit": "fahrenheit",
        "windspeed_unit": "mph",
        "timezone": "auto",
        "start_date": date,
        "end_date": date,
    })
    resp.raise_for_status()
    hourly = resp.json()["hourly"]
    hours = []
    for i, t in enumerate(hourly["time"]):
        hours.append({
            "time": t,
            "temp_f": hourly["temperature_2m"][i],
            "precip_probability_pct": hourly["precipitation_probability"][i],
            "wind_mph": hourly["windspeed_10m"][i],
            "conditions": WEATHER_CODES.get(hourly["weathercode"][i], "unknown"),
        })
    return {"hours": hours}

def get_air_quality_forecast(latitude: float, longitude: float, date: str) -> dict:
    """Get the air quality forecast (US AQI) for a specific date. Useful for outdoor exercise
    like hiking or running, since poor air quality (wildfire smoke, high pollution days) can
    matter as much as the weather itself — check this for any promising day before finalizing."""
    resp = requests.get("https://air-quality-api.open-meteo.com/v1/air-quality", params={
        "latitude": latitude, "longitude": longitude,
        "hourly": "us_aqi",
        "timezone": "auto",
        "start_date": date, "end_date": date,
    })
    resp.raise_for_status()
    values = [v for v in resp.json().get("hourly", {}).get("us_aqi", []) if v is not None]
    if not values:
        return {"date": date, "note": "No air quality data available for this date/location."}
    max_aqi = max(values)
    avg_aqi = round(sum(values) / len(values), 1)
    return {"date": date, "max_us_aqi": max_aqi, "avg_us_aqi": avg_aqi, "category": _aqi_label(max_aqi)}

def get_historical_baseline(latitude: float, longitude: float, month: int, day: int) -> dict:
    """Get the typical (historical average) max temperature and rainfall for this calendar
    month/day, based on the past 5 years of real historical weather data. Use this to tell the
    user whether the current forecast is unusually hot/cold/wet compared to normal for this time
    of year at this location — genuinely useful color for a final recommendation."""
    import datetime
    current_year = datetime.datetime.now().year
    temps, precips = [], []
    for years_back in range(1, 6):
        year = current_year - years_back
        date_str = f"{year:04d}-{int(month):02d}-{int(day):02d}"
        try:
            resp = requests.get("https://archive-api.open-meteo.com/v1/archive", params={
                "latitude": latitude, "longitude": longitude,
                "start_date": date_str, "end_date": date_str,
                "daily": "temperature_2m_max,precipitation_sum",
                "temperature_unit": "fahrenheit",
                "timezone": "auto",
            })
            resp.raise_for_status()
            daily = resp.json().get("daily", {})
            if daily.get("temperature_2m_max"):
                t = daily["temperature_2m_max"][0]
                p = daily["precipitation_sum"][0]
                if t is not None:
                    temps.append(t)
                if p is not None:
                    precips.append(p)
        except requests.HTTPError:
            continue
    if not temps:
        return {"note": "Historical data unavailable for this location/date."}
    return {
        "month": int(month), "day": int(day),
        "years_used": len(temps),
        "typical_avg_max_temp_f": round(sum(temps) / len(temps), 1),
        "typical_avg_precip_in": round(sum(precips) / len(precips), 2) if precips else None,
    }

def score_day_for_activity(date: str, day_max_temp_f: float, day_min_temp_f: float, day_precip_probability_pct: float, day_max_wind_mph: float, activity_type: str, max_temp_f: float = None, min_temp_f: float = None, max_precip_probability_pct: float = None, max_wind_mph: float = None, day_max_aqi: float = None, max_aqi: float = None) -> dict:
    """Check whether one day's forecast meets the given constraints for an activity. Pass that
    day's own values (read from get_daily_forecast / get_air_quality_forecast output) as
    individual numbers, not a whole object. Pass only the constraint parameters that actually
    matter for this goal; leave the rest as None."""
    reasons = []
    fits = True
    if max_temp_f is not None and day_max_temp_f > max_temp_f:
        fits = False
        reasons.append(f"too hot: {day_max_temp_f}°F > {max_temp_f}°F")
    if min_temp_f is not None and day_min_temp_f < min_temp_f:
        fits = False
        reasons.append(f"too cold: {day_min_temp_f}°F < {min_temp_f}°F")
    if max_precip_probability_pct is not None and day_precip_probability_pct > max_precip_probability_pct:
        fits = False
        reasons.append(f"too much rain risk: {day_precip_probability_pct}% > {max_precip_probability_pct}%")
    if max_wind_mph is not None and day_max_wind_mph > max_wind_mph:
        fits = False
        reasons.append(f"too windy: {day_max_wind_mph}mph > {max_wind_mph}mph")
    if max_aqi is not None and day_max_aqi is not None and day_max_aqi > max_aqi:
        fits = False
        reasons.append(f"air quality too poor: AQI {day_max_aqi} > {max_aqi}")
    return {"date": date, "activity_type": activity_type, "fits_criteria": fits, "reasons_if_not_fitting": reasons}

def suggest_alternative_activity(date: str, day_max_temp_f: float, day_precip_probability_pct: float, day_max_wind_mph: float) -> dict:
    """Given a day's forecast that didn't fit the originally requested activity, suggest what
    activity type WOULD suit those conditions instead. Pass the date and that day's own values as
    individual numbers, not a whole object."""
    if day_precip_probability_pct >= 50:
        suggestion = "indoor activity (climbing gym, museum) — rain risk too high for outdoor plans"
    elif day_max_temp_f >= 90:
        suggestion = "early morning or evening light activity — too hot for a midday outdoor session"
    elif day_max_wind_mph >= 25:
        suggestion = "sheltered/low-wind activity (forested trail, indoor court sports) — too windy for exposed activities"
    elif day_max_temp_f <= 40:
        suggestion = "shorter, higher-intensity activity to stay warm (run, bike) rather than a long slow hike"
    else:
        suggestion = "conditions are actually fine for most outdoor activities"
    return {"date": date, "suggested_alternative": suggestion}

In [4]:
from google import genai
from google.genai import types
import time

client = genai.Client(api_key=GOOGLE_AI_API_KEY)

TOOL_FUNCTIONS = [
    resolve_location,
    get_daily_forecast,
    get_hourly_forecast_for_day,
    get_air_quality_forecast,
    get_historical_baseline,
    score_day_for_activity,
    suggest_alternative_activity,
]

SYSTEM_PROMPT = """You are an autonomous outdoor-expedition-planning agent. You are given ONE
goal: find the best day (and possibly best location and/or activity) for an outdoor plan. You
must work through this completely on your own — do not ask the user questions or wait for
guidance mid-task.

Your process:
1. Resolve every candidate location mentioned to coordinates with resolve_location. If given
   several candidate locations, check ALL of them — don't just default to the first one.
2. For each location, pull the daily forecast with get_daily_forecast.
3. For promising days, also check get_air_quality_forecast — poor air quality matters for
   physical activity as much as weather does.
4. Use score_day_for_activity on each day (pass that day's own values as individual numbers) to
   check which ones meet the stated constraints, including air quality if relevant.
5. If NO day/location combination fits, do not give up and do not ask the user what to do —
   autonomously try, in this order: (a) pull a longer forecast window, (b) check other candidate
   locations if any were given, (c) loosen exactly ONE constraint at a time and clearly note
   which one and why, (d) if still nothing works, call suggest_alternative_activity on the single
   best-available day and recommend that instead.
6. Once you have a day that fits (or your best fallback), ALWAYS self-verify it with
   get_hourly_forecast_for_day before finalizing — a good daily average can hide a bad window
   within the day. Also check the day's sunrise/sunset times (from get_daily_forecast) and never
   recommend a start time before sunrise or after sunset.
7. Call get_historical_baseline for the winning day's month/day to add useful color — e.g. "this
   is 9°F warmer than a typical day here in mid-July."
8. Be efficient with tool calls — you have a limited number of tool-call turns available. Don't
   re-check the same location/date combination more than once, and don't call
   get_air_quality_forecast or get_hourly_forecast_for_day for days you've already ruled out on
   temperature/rain/wind alone.
9. Finish with a clear final recommendation: the exact day, location, and time window, the key
   numbers behind it, the historical comparison, and a short honest account of what you tried
   along the way — including any constraint you loosened or alternative you ended up suggesting.
"""

# NOTE: automatic function calling in this SDK defaults to a cap of 10 remote tool-call turns per
# message, to prevent infinite loops. A rich multi-location, multi-tool investigation like this one
# can genuinely need more than that, so we raise it explicitly. Keep in mind: each unit here counts
# toward your daily (RPD) quota, so a single goal can now cost up to 30 requests — iterate with
# simple single-location goals during development and save rich multi-location runs for your demo.
GENERATE_CONFIG = types.GenerateContentConfig(
    tools=TOOL_FUNCTIONS,
    system_instruction=SYSTEM_PROMPT,
    automatic_function_calling=types.AutomaticFunctionCallingConfig(maximum_remote_calls=30),
)

MODEL_NAME = "gemini-2.5-flash"

def _looks_like_transient_error(exc: Exception) -> bool:
    """Detect a retry-worthy error: either a 429 rate-limit/quota error, or a 503 'model
    overloaded / high demand' server error. Both are temporary and worth backing off and
    retrying; anything else (bad request, auth error, etc.) is a real bug and should raise."""
    text = f"{type(exc).__name__} {exc}".lower()
    rate_limit_markers = ["429", "resourceexhausted", "quota", "rate limit"]
    overload_markers = ["503", "unavailable", "overloaded", "high demand"]
    return any(marker in text for marker in rate_limit_markers + overload_markers)

def _is_rate_limit_specifically(exc: Exception) -> bool:
    """Narrower check used only to pick which final message to show after retries are exhausted."""
    text = f"{type(exc).__name__} {exc}".lower()
    return any(marker in text for marker in ["429", "resourceexhausted", "quota", "rate limit"])

def _run_agent_generator(goal_message: str, max_retries: int = 5, base_delay_seconds: int = 15):
    """Core generator. Yields ('status', message, None) for progress updates while retrying, and
    finally ('done', trace_text, final_answer_text) exactly once. Both the notebook-print version
    and the Gradio UI consume this same generator, so retry/rate-limit behavior only lives in one
    place."""
    for attempt in range(1, max_retries + 1):
        chat = client.chats.create(model=MODEL_NAME, config=GENERATE_CONFIG)
        try:
            response = chat.send_message(goal_message)
        except Exception as e:
            if _looks_like_transient_error(e) and attempt < max_retries:
                wait_seconds = base_delay_seconds * (2 ** (attempt - 1))
                yield ("status", f"Rate limited (attempt {attempt}/{max_retries}). Waiting {wait_seconds}s before retrying...", None)
                time.sleep(wait_seconds)
                continue
            elif _looks_like_transient_error(e):
                if _is_rate_limit_specifically(e):
                    msg = ("Still rate limited after all retries. If this is happening within the same "
                           "minute, wait ~60s and try again — that's the per-minute (RPM) limit. If it "
                           "happens on the very first call of a fresh session, you've likely hit the daily "
                           "(RPD) limit, which only resets at a fixed clock time (midnight Pacific) — "
                           "no amount of retrying will fix that until then.")
                else:
                    msg = ("The model is still reporting high demand / unavailable (a 503) after all "
                           "retries. This is Google's server capacity, not your quota — it usually "
                           "clears within a few minutes. Just try again shortly.")
                yield ("done", "(no tool calls completed — blocked by a temporary error)", msg)
                return
            else:
                raise

        trace_lines = []
        try:
            history = chat.get_history()
        except Exception:
            history = []
        for entry in history:
            parts = getattr(entry, "parts", None) or []
            for part in parts:
                fc = getattr(part, "function_call", None)
                if fc and getattr(fc, "name", None):
                    args = _to_plain(dict(fc.args)) if fc.args else {}
                    trace_lines.append(f"→ {fc.name}({args})")
                fr = getattr(part, "function_response", None)
                if fr and getattr(fr, "name", None):
                    result = _to_plain(dict(fr.response)) if fr.response else {}
                    trace_lines.append(f"   result: {result}")
        trace_text = "\n".join(trace_lines) if trace_lines else "(no tool calls were made, or history wasn't available)"

        if not response.text:
            msg = ("The agent ran out of its tool-call budget or the model produced no final answer. "
                   "Try raising maximum_remote_calls further, or simplify the goal (e.g. fewer "
                   "candidate locations) and try again.")
            yield ("done", trace_text, msg)
            return

        yield ("done", trace_text, response.text)
        return

def run_autonomous_search(goal_message: str, max_retries: int = 5, base_delay_seconds: int = 15):
    """Notebook-friendly wrapper: prints progress as it happens, returns (trace_text, final_answer_text)."""
    for kind, a, b in _run_agent_generator(goal_message, max_retries, base_delay_seconds):
        if kind == "status":
            print(a + "\n")
        else:
            print("=== Agent's autonomous investigation ===\n")
            print(a)
            print("\n=== Final recommendation ===\n")
            print(b)
            return a, b


In [5]:
goal = (
    "Compare Denver, Colorado and Boulder, Colorado. Find me the best day in the next 7 days for "
    "a hike. I want max temp under 80F, less than 30% chance of rain, wind under 20mph, and good "
    "or moderate air quality. If nothing fits exactly, figure out the closest option on your own "
    "and tell me what you had to compromise on."
)

run_autonomous_search(goal)

=== Agent's autonomous investigation ===

→ resolve_location({'place_name': 'Denver, Colorado'})
→ resolve_location({'place_name': 'Boulder, Colorado'})
   result: {'result': {'matches': [{'name': 'Denver', 'country': 'United States', 'admin1': 'Colorado', 'latitude': 39.73915, 'longitude': -104.9847}, {'name': 'Denver City', 'country': 'United States', 'admin1': 'Texas', 'latitude': 32.96455, 'longitude': -102.8291}, {'name': 'Denver', 'country': 'United States', 'admin1': 'Pennsylvania', 'latitude': 40.23315, 'longitude': -76.13717}, {'name': 'Denver', 'country': 'United States', 'admin1': 'North Carolina', 'latitude': 35.53125, 'longitude': -81.0298}, {'name': 'Denver', 'country': 'United States', 'admin1': 'Iowa', 'latitude': 42.67137, 'longitude': -92.3374}], 'note': "No results for 'Denver, Colorado' as given; retried successfully with 'Denver'."}}
   result: {'result': {'matches': [{'name': 'Boulder', 'country': 'United States', 'admin1': 'Colorado', 'latitude': 40.01499, 'longi

('→ resolve_location({\'place_name\': \'Denver, Colorado\'})\n→ resolve_location({\'place_name\': \'Boulder, Colorado\'})\n   result: {\'result\': {\'matches\': [{\'name\': \'Denver\', \'country\': \'United States\', \'admin1\': \'Colorado\', \'latitude\': 39.73915, \'longitude\': -104.9847}, {\'name\': \'Denver City\', \'country\': \'United States\', \'admin1\': \'Texas\', \'latitude\': 32.96455, \'longitude\': -102.8291}, {\'name\': \'Denver\', \'country\': \'United States\', \'admin1\': \'Pennsylvania\', \'latitude\': 40.23315, \'longitude\': -76.13717}, {\'name\': \'Denver\', \'country\': \'United States\', \'admin1\': \'North Carolina\', \'latitude\': 35.53125, \'longitude\': -81.0298}, {\'name\': \'Denver\', \'country\': \'United States\', \'admin1\': \'Iowa\', \'latitude\': 42.67137, \'longitude\': -92.3374}], \'note\': "No results for \'Denver, Colorado\' as given; retried successfully with \'Denver\'."}}\n   result: {\'result\': {\'matches\': [{\'name\': \'Boulder\', \'country